In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from loguru import logger
import os

print("Current working directory:", os.getcwd())

# Configure loguru for notebook display
# logger.remove()
# logger.add(lambda msg: print(msg, end=''), colorize=True, format="{level.icon} {message}")

# Data paths
ANNOT_FILE = "data/dataset_092624.xlsx"
OUTPUT_FILE = "data/dataset_092624_validated.xlsx"

Current working directory: c:\Users\beav3503\dev\llm_metadata


## 1. Load Raw Data

In [ ]:
# Load the raw Excel file
raw_df = pd.read_excel(ANNOT_FILE)

print(f"Loaded {len(raw_df)} rows, {len(raw_df.columns)} columns")
print(f"\nColumns: {list(raw_df.columns)}")
print(f"\nSource distribution:")
print(raw_df['source'].value_counts())
print(f"\nTotal valid (valid_yn='yes'): {(raw_df['valid_yn'] == 'yes').sum()}")
raw_df.head()

## 1.5 Inspect URL Fields for SS Records

In [ ]:
import re

# Separate records by source
ss_df = raw_df[raw_df['source'] == 'semantic_scholar'].copy()
dryad_zenodo_df = raw_df[raw_df['source'].isin(['dryad', 'zenodo'])].copy()

print(f"Semantic Scholar records: {len(ss_df)}")
print(f"Dryad/Zenodo records:     {len(dryad_zenodo_df)}")

# Inspect url vs url.1 for SS records
url_same = (ss_df['url'] == ss_df['url.1']).sum()
url_both_null = (ss_df['url'].isna() & ss_df['url.1'].isna()).sum()
url_diff = len(ss_df) - url_same - url_both_null

print(f"\n--- SS URL column comparison ---")
print(f"url == url.1 (identical): {url_same}")
print(f"Both null:                {url_both_null}")
print(f"Different:                {url_diff}")

# Show sample
print(f"\n--- SS url sample (first 10) ---")
print(ss_df[['url', 'url.1']].head(10).to_string())

print(f"\n--- Dryad/Zenodo url sample (first 5) ---")
print(dryad_zenodo_df[['url', 'url.1']].head(5).to_string())

## 2. Run Validation

In [ ]:
from llm_metadata.schemas import DataFrameValidator, ValidationReport
from llm_metadata.schemas.fuster_features import DatasetFeaturesNormalized

# Validate ALL records (Dryad + Zenodo + SS) through updated DatasetFeaturesNormalized
# This includes WU-A1 modulator fields: time_series, multispecies, threatened_species,
# new_species_science, new_species_region, bias_north_south
validator = DataFrameValidator(model=DatasetFeaturesNormalized, strict=False)
report = validator.validate(raw_df)

print(report.summary())

# Source-level breakdown
print("\nBreakdown by source:")
for source in raw_df['source'].value_counts().index:
    source_idx = set(raw_df[raw_df['source'] == source].index)
    n_valid = len([i for i in report.valid_indices if i in source_idx])
    n_invalid = len([i for i in report.invalid_indices if i in source_idx])
    print(f"  {source:<20}: {n_valid} valid, {n_invalid} invalid")

## 3. Review Errors

In [5]:
# Get errors as DataFrame for inspection
errors_df = report.errors_to_dataframe()

if len(errors_df) > 0:
    print(f"Total errors: {len(errors_df)}")
    print(f"\nError distribution by field:")
    print(errors_df['field'].value_counts())
    print(f"\nError distribution by type:")
    print(errors_df['error_type'].value_counts())
else:
    print("✅ No errors found!")

errors_df.head(20)

✅ No errors found!


,row_index,field,raw_value,error_type,message,suggested_fix


In [6]:
# View errors with suggested fixes
if len(errors_df) > 0:
    errors_with_fixes = errors_df[errors_df['suggested_fix'] != ''][['row_index', 'field', 'raw_value', 'suggested_fix']]
    print(f"Errors with suggested fixes: {len(errors_with_fixes)}")
    display(errors_with_fixes)

## 4. Review Invalid Rows

In [7]:
# Get invalid rows for correction
invalid_df = report.invalid_rows_to_dataframe()

# Select only relevant columns for display
relevant_cols = ['data_type', 'geospatial_info_dataset', 'spatial_range_km2', 
                 'temporal_range', 'temp_range_i', 'temp_range_f', 'taxons', 'referred_dataset']
available_cols = [c for c in relevant_cols if c in invalid_df.columns]

print(f"Invalid rows: {len(invalid_df)}")
if len(invalid_df) > 0:
    display(invalid_df[available_cols].head(10))

Invalid rows: 0


In [ ]:
# --- SPECIES VALIDATION CHECK ---
print("Checking species field transformation...")

# Re-run validation with DatasetFeaturesNormalized to get validated rows
validator = DataFrameValidator(model=DatasetFeaturesNormalized, strict=False)
report = validator.validate(raw_df)
validated_df = report.valid_rows_to_dataframe()

# Rows that have species info
params_with_species = raw_df[raw_df['species'].notna()].index
print(f"Rows with species: {len(params_with_species)}")

if len(params_with_species) > 0:
    sample_indices = params_with_species[:50]

    print("\nComparing Raw vs Validated Species (First 50 examples):")
    comparison = pd.DataFrame({
        'Raw Species': raw_df.loc[sample_indices, 'species'],
        'Validated Species': validated_df.reindex(raw_df.index).loc[sample_indices, 'species']
    })

    # Check type of validated species
    valid_types = comparison['Validated Species'].apply(type).unique()
    print(f"Validated column types: {valid_types}")

    display(comparison)
else:
    print("No species data found in raw dataframe.")

## 5. Coverage Stats by Source

In [ ]:
def has_doi(url):
    """Return True if the URL contains a DOI."""
    if pd.isna(url) or not url:
        return False
    return bool(re.search(r'doi\.org/', str(url), re.IGNORECASE))


sources_to_check = ['dryad', 'zenodo', 'semantic_scholar', 'referenced']
stats_rows = []

for source in sources_to_check:
    src_df = raw_df[raw_df['source'] == source]
    if len(src_df) == 0:
        continue
    n_total = len(src_df)
    n_valid = int((src_df['valid_yn'] == 'yes').sum())
    n_abstract = int(src_df['full_text'].notna().sum())
    n_doi = int(src_df['url'].apply(has_doi).sum())
    n_cited = int(src_df['cited_articles'].notna().sum())
    stats_rows.append({
        'source': source,
        'total_records': n_total,
        'valid_records': n_valid,
        'has_abstract': n_abstract,
        'abstract_pct': f"{100*n_abstract/n_total:.1f}%",
        'has_doi': n_doi,
        'doi_pct': f"{100*n_doi/n_total:.1f}%",
        'has_cited_articles': n_cited,
    })

coverage_df = pd.DataFrame(stats_rows)
print("Coverage Stats by Source:")
display(coverage_df)

print(f"\nTotals (all sources):")
print(f"  Total records:          {len(raw_df)}")
print(f"  Valid (valid_yn='yes'): {(raw_df['valid_yn'] == 'yes').sum()}")
print(f"  Has abstract:           {raw_df['full_text'].notna().sum()}")

## 6. Filter to Valid Records

In [ ]:
# Filter schema-validated rows to those where valid_yn == 'yes'
# (biodiversity-relevant records only)
schema_valid_df = report.valid_rows_to_dataframe()
valid_records_df = schema_valid_df[schema_valid_df['valid_yn'] == 'yes'].copy()

print(f"Schema-valid rows:             {len(schema_valid_df)}")
print(f"Schema-valid + valid_yn='yes': {len(valid_records_df)}")

print(f"\nValid records by source:")
print(valid_records_df['source'].value_counts())

## 7. Stats Table for Presentation

In [ ]:
print("=== DATASET COVERAGE STATS FOR PRESENTATION ===\n")
header = f"{'Source':<22} {'Records':>9} {'Valid':>7} {'Abstract':>10} {'Has DOI':>9} {'Cited Art.':>11}"
sep = "-" * len(header)
print(header)
print(sep)

presentation_sources = ['dryad', 'zenodo', 'semantic_scholar']
totals = {'total': 0, 'valid': 0, 'abstract': 0, 'doi': 0, 'cited': 0}

for source in presentation_sources:
    src_df = raw_df[raw_df['source'] == source]
    n_total = len(src_df)
    n_valid = int((src_df['valid_yn'] == 'yes').sum())
    n_abstract = int(src_df['full_text'].notna().sum())
    n_doi = int(src_df['url'].apply(has_doi).sum())
    n_cited = int(src_df['cited_articles'].notna().sum())
    print(f"{source:<22} {n_total:>9} {n_valid:>7} {n_abstract:>10} {n_doi:>9} {n_cited:>11}")
    totals['total'] += n_total
    totals['valid'] += n_valid
    totals['abstract'] += n_abstract
    totals['doi'] += n_doi
    totals['cited'] += n_cited

print(sep)
print(f"{'TOTAL':<22} {totals['total']:>9} {totals['valid']:>7} {totals['abstract']:>10} {totals['doi']:>9} {totals['cited']:>11}")
print()

# Validation error breakdown by source
print("=== VALIDATION ERROR BREAKDOWN BY SOURCE ===\n")
errors_df = report.errors_to_dataframe()
if len(errors_df) > 0:
    for source in presentation_sources:
        src_idx = set(raw_df[raw_df['source'] == source].index)
        src_errors = errors_df[errors_df['row_index'].isin(src_idx)]
        print(f"{source}: {len(src_errors)} errors")
        if len(src_errors) > 0:
            print(src_errors['field'].value_counts().to_string())
else:
    print("No validation errors found across all sources.")

## 7. Export Validated Data

In [ ]:
# valid_records_df: schema-valid rows filtered to valid_yn == 'yes'
# (computed in Section 6 above)
print(f"Records to export: {len(valid_records_df)}")
print(f"\nBy source:")
print(valid_records_df['source'].value_counts())
valid_records_df.head()

In [ ]:
# Export valid records (schema-valid + valid_yn='yes') to OUTPUT_FILE
valid_records_df.to_excel(OUTPUT_FILE, index=False)
print(f"Exported {len(valid_records_df)} validated records to {OUTPUT_FILE}")
print(f"\nSummary: {len(valid_records_df)} valid records from {len(raw_df)} total")
print(f"  Dryad:            {(valid_records_df['source'] == 'dryad').sum()}")
print(f"  Zenodo:           {(valid_records_df['source'] == 'zenodo').sum()}")
print(f"  Semantic Scholar: {(valid_records_df['source'] == 'semantic_scholar').sum()}")